### Create Evaluation Golden DataSets 

In [ ]:
from langsmith import Client
import warnings
from urllib3.exceptions import NotOpenSSLWarning
warnings.filterwarnings("ignore", category=NotOpenSSLWarning)


client = Client()

database_name = "PDF-Rag-Evaluation"

database = client.create_dataset(database_name)

# Create Actual Facts Answer In Golden Dataset
client.create_examples(
    dataset_id=database.id,
    examples=[
        {
            "inputs": {"question": "Which objects are luminous?"},
            "outputs": {"answer": "The Sun, stars, fire, and lightning."},
        },
        {
            "inputs": {"question": "Is the Moon luminous?"},
            "outputs": {"answer": "No, it reflects sunlight and is non-luminous."},
        },
        {
            "inputs": {"question": "How does light travel?"},
            "outputs": {"answer": "Light travels in a straight line."},
        },
        {
            "inputs": {"question": "What happens when an opaque object blocks light?"},
            "outputs": {"answer": "A shadow is formed on the screen."},
        },
        {
            "inputs": {"question": "Do translucent objects form shadows?"},
            "outputs": {"answer": "Yes, they form lighter shadows."},
        },
        {
            "inputs": {"question": "What is reflection of light?"},
            "outputs": {"answer": "It is the change in direction of light by a shiny surface or mirror."},
        },
        {
            "inputs": {"question": "What kind of image does a plane mirror form?"},
            "outputs": {"answer": "An erect, same-sized, laterally inverted image."},
        },
        {
            "inputs": {"question": "Can a plane mirror image be obtained on a screen?"},
            "outputs": {"answer": "No, it cannot be obtained on a screen."},
        },
        {
            "inputs": {"question": "What image does a pinhole camera produce?"},
            "outputs": {"answer": "An inverted image of the object on the screen."},
        },
        {
            "inputs": {"question": "What is lateral inversion?"},
            "outputs": {"answer": "It is the left-right reversal seen in mirror images."},
        }
    ]
)


### LLM as Judge - Evaluator

In [ ]:
from langchain_groq import ChatGroq
from config import VECTORSTORE_PATH
from app.llm_chain import build_chain
from app.embeddings import embeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv
import os

load_dotenv()

print("VECTORSTORE_PATH----:", VECTORSTORE_PATH)

#llama-3.1-70b-versatile
llm_As_Judge = ChatGroq(model="openai/gpt-oss-120b", temperature=0.0)

if os.path.exists(VECTORSTORE_PATH):
    vectorstore = Chroma(
        persist_directory=VECTORSTORE_PATH,
        embedding_function=embeddings
    )
    chain = build_chain(vectorstore)
else:
    raise FileNotFoundError("Run ingest.py first")


# ================================================================================
# Run Your actuall Rag Pipeline
def ls_target(input: dict) -> dict:
    result= chain(input['question'])
    return {
        "response": result["response"],
        "documents": result["documents"]
        }

#Evaluator -: Seperate LLM Call not rag pipeline
def correctness(inputs:dict, outputs:dict, reference_outputs: dict)->dict:
    # Clean up whitespace
    predicted = outputs['response'].strip()
    reference = reference_outputs['answer'].strip()
    
    prompt = f"""Question: {inputs['question']}
        Reference Answer: {reference}
        Predicted Answer: {predicted}

        Compare the Predicted Answer to the Reference Answer.
        Reply with only one word: "Correct" if they match, otherwise "Incorrect".
        """
    
    # Call LLM as gudge to check score
    grade = llm_As_Judge.invoke(prompt).content.strip().lower()
    score = 1 if grade.startswith("correct") else 0
    label = "Correct" if score else "Incorrect"
    
    return {"key": "correctness", "score": score, "label": label}

# relevance evaluator --:
def relevance(inputs: dict, outputs: dict) -> dict:
    """Binary relevance evaluator for RAG retrieval"""
    predicted = outputs['response'].strip()
    
    prompt = f"""You are evaluating whether a Rag LLM Response is relevant to a given query. 
    A document is relevant if it contains information that directly helps answer the query. 
    If it does not, mark it as irrelevant.

    Return only one of:
    - "Relevant"
    - "Irrelevant"

    Query: {inputs['question']}
    Document: {predicted}
    """
    
    # Call LLM as judge
    grade = llm_As_Judge.invoke(prompt).content.strip().lower()
    
    # Binary relevance check
    score = "relevant" in grade
    label = "Relevant" if score else "Irrelevant"
    
    
    return {"key": "relevance", "score": score, "label": label}


# Groundedness -: Response vs Retrive Docs
def groundedness(inputs: dict, outputs: dict) -> dict:
    """A simple evaluator for RAG Answer Groundedness"""
    
    # Combine retrieved docs
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = outputs["response"].strip()
     
    prompt = f"""You are evaluating whether the answer is grounded in the provided documents. 
    An answer is grounded if the facts it states can be verified in the retrieved documents. 
    If the answer contains information not supported by the documents, mark it as not grounded.

    Return only one of:
    - "Grounded"
    - "Not Grounded"

    Documents:
    {doc_string}

    Answer:
    {answer}
    """
    
    # Call LLM as Judge
    result = llm_As_Judge.invoke(prompt).content.strip().lower()
    
    # Binary groundedness check
    score = "grounded" in result
    label = "Grounded" if score else "Not Grounded"

    
    return {"key": "groundedness", "score": score, "label": label}


# Retrieval Relevance -: Retrieval Docs vs Input
def retrieval_relevance(inputs: dict, outputs: dict) -> dict:
    """Binary retrieval relevance evaluator for RAG retrieval relevance"""
    # get retrieved docs
    retrieved_docs = "\n\n".join(doc.page_content for doc in outputs["documents"])
    
    prompt = f"""You are evaluating whether a retrieved document is relevant to a given query. 
    A document is relevant if it contains information that directly helps answer the query. 
    If it does not, mark it as irrelevant.

    Return only one of:
    - "Relevant"
    - "Irrelevant"

    Query: {inputs['question']}
    Document: {retrieved_docs}
    """
    
    # Call LLM as judge
    grade = llm_As_Judge.invoke(prompt).content.strip().lower()
    
    # Binary relevance check
    score = "relevant" in grade
    label = "Relevant" if score else "Irrelevant"
    
    return {"key": "retrieval_relevance", "score": score, "label":label}
         
     


### Run Evaluation -:

In [ ]:
from langsmith import Client

client= Client()

database_name = "PDF-Rag-Evaluation"

experiment_result = client.evaluate(
    ls_target,
    data=database_name,
    evaluators=[correctness,relevance,groundedness,retrieval_relevance],
    experiment_prefix="PDF_Doc_Evaluatio",
)

experiment_result.to_pandas()